# Applied Machine Learning — Titanic

End-to-end supervised learning workflow: data inspection, cleaning, feature engineering, Bernoulli Naive Bayes, Linear Regression, Ridge, LASSO, and model evaluation.

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer

from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LinearRegression, Ridge, Lasso

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Load the Data

In [ ]:
train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


## 3. View the Raw Dataset

In [ ]:
display(train.head())
display(test.head())


In [ ]:
print("TRAIN INFO")
train.info()


In [ ]:
print("TRAIN SUMMARY")
display(train.describe(include="all"))


## 4. Initial Data Quality Check

In [ ]:
print("Missing values in train:")
display(train.isnull().sum())

print("\nMissing values in test:")
display(test.isnull().sum())

print("\nDuplicate rows:")
print("Train:", train.duplicated().sum())
print("Test:", test.duplicated().sum())

print("\nNumber of unique values:")
display(train.nunique())


In [ ]:
for col in ["Pclass", "Sex", "Embarked"]:
    print(f"\n{col}:")
    display(train[col].value_counts(dropna=False))


## 5. Data Cleaning

Cleaning is performed before the train/validation split for operations that do not learn parameters from the data. Missing-value imputation, scaling, and encoding are handled later inside model-specific pipelines so that those parameters are learned from the training split only.

In [ ]:
# Standardize whitespace and capitalization in text fields
for df in [train, test]:
    for col in ["Name", "Sex", "Ticket", "Cabin", "Embarked"]:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )

    df["Sex"] = df["Sex"].str.lower()
    df["Embarked"] = df["Embarked"].str.upper()

# Replace impossible numerical values with missing values
for df in [train, test]:
    df.loc[df["Age"] < 0, "Age"] = np.nan
    df.loc[df["Fare"] < 0, "Fare"] = np.nan
    df.loc[df["SibSp"] < 0, "SibSp"] = np.nan
    df.loc[df["Parch"] < 0, "Parch"] = np.nan

print("Cleaning completed.")
print("Train shape after cleaning:", train.shape)
print("Test shape after cleaning:", test.shape)


## 6. Train / Validation Split

In [ ]:
X = train.drop(columns=["Survived"])
y = train["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)


## 7. Training-Set Exploration

In [ ]:
print("Training data information:")
X_train.info()


In [ ]:
print("Missing values in training predictors:")
display(X_train.isnull().sum())

print("\nMissing percentages:")
display((X_train.isnull().mean() * 100).round(2))


In [ ]:
print("Duplicate rows in X_train:", X_train.duplicated().sum())


### Categorical distributions

In [ ]:
categorical_exploration = ["Pclass", "Sex", "Embarked"]

for col in categorical_exploration:
    counts = X_train[col].value_counts(dropna=False)
    percentages = X_train[col].value_counts(
        dropna=False, normalize=True
    ).mul(100).round(2)

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages
    })

    print(f"\n{col}")
    display(summary)

    plt.figure(figsize=(6, 4))
    ax = sns.countplot(data=X_train, x=col)
    plt.title(f"{col} Distribution")

    total = len(X_train)
    for container in ax.containers:
        labels = [
            f"{bar.get_height():.0f}\n({bar.get_height()/total*100:.1f}%)"
            for bar in container
        ]
        ax.bar_label(container, labels=labels)

    plt.tight_layout()
    plt.show()


### Numerical skewness

In [ ]:
skew_features = ["Age", "Fare", "SibSp", "Parch"]

skewness = X_train[skew_features].skew().sort_values()

display(
    skewness.to_frame("Skewness")
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.ravel(), skew_features):
    sns.histplot(X_train[col], kde=True, ax=ax)
    ax.set_title(f"{col} Distribution")

plt.tight_layout()
plt.show()


### Target distribution

In [ ]:
target_counts = y_train.value_counts().sort_index()
target_percentages = y_train.value_counts(
    normalize=True
).sort_index().mul(100).round(2)

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percentages
})

display(target_summary)

plt.figure(figsize=(6, 4))
ax = sns.countplot(x=y_train)
plt.title("Training Target Distribution")
plt.xlabel("Survived")
plt.ylabel("Count")

for container in ax.containers:
    labels = [
        f"{bar.get_height():.0f}\n({bar.get_height()/len(y_train)*100:.1f}%)"
        for bar in container
    ]
    ax.bar_label(container, labels=labels)

plt.tight_layout()
plt.show()


## 8. Feature Engineering

The engineered features capture useful structure from the raw Titanic variables. Fare is log-transformed because it is strongly right-skewed. FamilySize combines family counts, IsAlone captures the special case of a passenger traveling without family, Title summarizes Name, Deck summarizes Cabin location, and TicketPrefix captures ticket-group information.

In [ ]:
def feature_engineer(df):
    df = df.copy()

    # Log-transform strongly right-skewed Fare
    df["FareLog"] = np.log1p(df["Fare"])

    # Family-related features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Extract and group passenger title
    df["Title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
    )

    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    df["Title"] = df["Title"].where(
        df["Title"].isin(["Mr", "Miss", "Mrs", "Master"]),
        "Rare"
    )

    # Extract cabin deck
    df["Deck"] = df["Cabin"].str[0].fillna("Unknown")

    # Extract ticket prefix
    df["TicketPrefix"] = (
        df["Ticket"]
        .str.replace(r"\d", "", regex=True)
        .str.replace(r"[\s./]+", "", regex=True)
        .str.upper()
    )

    df["TicketPrefix"] = df["TicketPrefix"].replace(
        "", "NUMERIC"
    )

    return df


X_train_fe = feature_engineer(X_train)
X_val_fe = feature_engineer(X_val)
test_fe = feature_engineer(test)

print("Feature engineering completed.")


## 9. Prepare Final Model Features

PassengerId is an identifier. Name, Ticket, and Cabin have already been summarized into engineered features. SibSp and Parch are used to construct FamilySize and are excluded from the regression inputs to reduce redundant family predictors. The original Fare is replaced by FareLog.

In [ ]:
columns_to_drop = [
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin",
    "SibSp",
    "Parch",
    "Fare"
]

X_train_model = X_train_fe.drop(columns=columns_to_drop)
X_val_model = X_val_fe.drop(columns=columns_to_drop)
test_model = test_fe.drop(columns=columns_to_drop)

numerical_features = [
    "Age",
    "FareLog",
    "FamilySize"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
    "Deck",
    "TicketPrefix",
    "IsAlone"
]

print("Final model features:")
print(X_train_model.columns.tolist())


## 10. Regression Preprocessing

Linear Regression, Ridge, and LASSO use the same preprocessing: median imputation and standardization for numerical variables, and most-frequent imputation plus one-hot encoding for categorical variables.

In [ ]:
linear_numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

linear_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

regression_preprocessor = ColumnTransformer([
    ("num", linear_numerical_pipeline, numerical_features),
    ("cat", linear_categorical_pipeline, categorical_features)
])


## 11. Bernoulli Naive Bayes Preprocessing

BernoulliNB works with binary-style features. Numerical predictors are therefore imputed and discretized into five one-hot bins, while categorical predictors are imputed and one-hot encoded.

In [ ]:
nb_numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("bins", KBinsDiscretizer(
        n_bins=5,
        encode="onehot",
        strategy="quantile",
        subsample=None
    ))
])

nb_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

nb_preprocessor = ColumnTransformer([
    ("num", nb_numerical_pipeline, numerical_features),
    ("cat", nb_categorical_pipeline, categorical_features)
])

X_train_nb_processed = nb_preprocessor.fit_transform(
    X_train_model
)

X_val_nb_processed = nb_preprocessor.transform(
    X_val_model
)

print("NB training shape:", X_train_nb_processed.shape)
print("NB validation shape:", X_val_nb_processed.shape)


## 12. Bernoulli Naive Bayes: Smoothing Comparison

In [ ]:
def evaluate_nb(alpha):
    model = BernoulliNB(alpha=alpha)

    model.fit(
        X_train_nb_processed,
        y_train
    )

    y_pred = model.predict(
        X_val_nb_processed
    )

    y_score = model.predict_proba(
        X_val_nb_processed
    )[:, 1]

    result = {
        "Model": "BernoulliNB",
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "AUC": (
            roc_auc_score(y_val, y_score)
            if np.isfinite(y_score).all()
            else np.nan
        )
    }

    return model, y_pred, y_score, result


alphas = [0, 0.01, 1.0, 10.0]

nb_models = {}
nb_predictions = {}
nb_probabilities = {}
nb_results = []

for alpha in alphas:
    model, y_pred, y_score, result = evaluate_nb(alpha)

    nb_models[alpha] = model
    nb_predictions[alpha] = y_pred
    nb_probabilities[alpha] = y_score
    nb_results.append(result)

nb_results_df = pd.DataFrame(nb_results)

display(nb_results_df)


### BernoulliNB confusion matrices

In [ ]:
fig, axes = plt.subplots(
    1, len(alphas), figsize=(16, 4)
)

for ax, alpha in zip(axes, alphas):
    cm = confusion_matrix(
        y_val,
        nb_predictions[alpha]
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["Did Not Survive", "Survived"],
        yticklabels=["Did Not Survive", "Survived"]
    )

    ax.set_title(f"BernoulliNB (α = {alpha})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 13. Linear Regression

In [ ]:
linear_model = Pipeline([
    ("preprocessor", regression_preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(
    X_train_model,
    y_train
)

linear_scores = linear_model.predict(
    X_val_model
)

linear_predictions = (
    linear_scores >= 0.5
).astype(int)


## 14. Ridge Regression

In [ ]:
ridge_model = Pipeline([
    ("preprocessor", regression_preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(
    X_train_model,
    y_train
)

ridge_scores = ridge_model.predict(
    X_val_model
)

ridge_predictions = (
    ridge_scores >= 0.5
).astype(int)


## 15. LASSO Regression

In [ ]:
lasso_model = Pipeline([
    ("preprocessor", regression_preprocessor),
    ("model", Lasso(alpha=0.01))
])

lasso_model.fit(
    X_train_model,
    y_train
)

lasso_scores = lasso_model.predict(
    X_val_model
)

lasso_predictions = (
    lasso_scores >= 0.5
).astype(int)


## 16. Store Regression Models

In [ ]:
regression_models = {
    "Linear Regression": {
        "model": linear_model,
        "predictions": linear_predictions,
        "scores": linear_scores
    },
    "Ridge": {
        "model": ridge_model,
        "predictions": ridge_predictions,
        "scores": ridge_scores
    },
    "LASSO": {
        "model": lasso_model,
        "predictions": lasso_predictions,
        "scores": lasso_scores
    }
}


## 17. Compare All Models

In [ ]:
evaluation_results = []

# BernoulliNB
for alpha in alphas:
    y_pred = nb_predictions[alpha]
    y_score = nb_probabilities[alpha]

    evaluation_results.append({
        "Model": f"BernoulliNB (α={alpha})",
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "AUC": (
            roc_auc_score(y_val, y_score)
            if np.isfinite(y_score).all()
            else np.nan
        )
    })

# Regression models
for name, data in regression_models.items():
    y_pred = data["predictions"]
    y_score = data["scores"]

    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "AUC": roc_auc_score(y_val, y_score)
    })

comparison_df = pd.DataFrame(evaluation_results)

comparison_df = comparison_df.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

display(comparison_df)


## 18. Regression Confusion Matrices

In [ ]:
fig, axes = plt.subplots(
    1, 3, figsize=(15, 4)
)

for ax, (name, data) in zip(
    axes,
    regression_models.items()
):
    cm = confusion_matrix(
        y_val,
        data["predictions"]
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["Did Not Survive", "Survived"],
        yticklabels=["Did Not Survive", "Survived"]
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 19. ROC Curves

In [ ]:
plt.figure(figsize=(9, 7))

# BernoulliNB
for alpha in alphas:
    y_score = nb_probabilities[alpha]

    if np.isfinite(y_score).all():
        fpr, tpr, _ = roc_curve(
            y_val,
            y_score
        )

        auc = roc_auc_score(
            y_val,
            y_score
        )

        plt.plot(
            fpr,
            tpr,
            label=f"BernoulliNB α={alpha} (AUC={auc:.3f})"
        )

# Regression models
for name, data in regression_models.items():
    y_score = data["scores"]

    fpr, tpr, _ = roc_curve(
        y_val,
        y_score
    )

    auc = roc_auc_score(
        y_val,
        y_score
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC={auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Model Comparison")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


## 20. Best Model

The comparison table above can be used to identify the best validation model. Accuracy, precision, recall, F1, and AUC should be considered together rather than relying on a single metric.

In [ ]:
best_model = comparison_df.iloc[0]

print("Top model by F1:")
display(best_model.to_frame().T)


## 21. Notes for the Report

- General data cleaning was performed before splitting because these operations do not estimate parameters from the validation data.
- Missing-value imputation, scaling, and encoding were fitted only on the training data through pipelines to prevent data leakage.
- Fare was log-transformed because it was strongly right-skewed.
- FamilySize was created from SibSp and Parch; the original family-count predictors were excluded from the regression inputs to reduce redundancy.
- IsAlone, Title, Deck, and TicketPrefix were treated as categorical features.
- BernoulliNB used discretized numerical variables because it expects binary-style inputs.
- Linear Regression predictions were converted to binary classes using the required 0.5 threshold. Ridge and LASSO used the same threshold.
- Alpha=0 represents no smoothing for BernoulliNB, while positive alpha values apply smoothing.